# Repeated-Seed Benchmark — tightening the C vs C* comparison

The single-seed run gave:

| Regime | ROC-AUC |
|---|---|
| A  new condition, known protein | 0.834 ± 0.009 |
| B  new protein | 0.783 ± 0.013 |
| C  new family (their groups) | 0.682 ± 0.031 |
| C* new family (homology-hardened) | 0.648 ± 0.040 |
| random-merge control | 0.686 ± 0.043 |

The C-C* gap (+0.034) is in the right direction and the control rules out group
coarsening as its cause, but at 5 folds it sits at t ~ 1.5 — not enough to state
as a result.

This notebook repeats the 5-fold CV under several random partitions of the same
groups, so the mean is estimated from 25 folds instead of 5. The means will barely
move; the error bars shrink.

**Seed budget** — A and B are already tight, so they get 2 seeds; the three
groupings under comparison get 5.

**Setup:** Runtime -> T4 GPU. Roughly 50-60 min.
Upload `llpsense_final_noseq.csv` and `llpsense_t5.npz`.

## 1. Setup and data

In [ ]:
import numpy as np, pandas as pd, subprocess, time, warnings, json
warnings.filterwarnings("ignore")
GPU = subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode == 0
print("GPU:", GPU)

from google.colab import files
# up = files.upload()        # llpsense_final_noseq.csv AND llpsense_t5.npz

df  = pd.read_csv("llpsense_final_noseq.csv")
z   = np.load("llpsense_t5.npz", allow_pickle=True)
emb = pd.DataFrame(z["X"], index=z["SeqID"])
COND = ['Conc','pH','temp','MgCl2','NaCl','KCl','PEG300-1k','PEG3k-6k','PEG8k-20k',
        'Ficoll','Dextran -40','Dextran 70-','Glycerol']
X = np.hstack([emb.loc[df.SeqID].to_numpy(np.float32), df[COND].to_numpy(np.float32)])
y, w = df.Label.to_numpy(), df.w_record.to_numpy()
print("features:", X.shape)

GPU: True
features: (50557, 1037)


## 2. Rebuild the random-merge control

Their 234 groups merged into 168 at random, using the **same merge-size pattern** as
the homology-hardened grouping but ignoring homology. This separates "fewer, coarser
groups" from "homologues removed".

In [ ]:
rec = df.drop_duplicates("RecordID")
pattern = sorted(rec.groupby("GroupX")["Group"].nunique().tolist(), reverse=True)
their = rec.Group.unique().tolist()
rng = np.random.default_rng(0); rng.shuffle(their)
mapping, k = {}, 0
for gi, size in enumerate(pattern):
    for g in their[k:k+size]: mapping[g] = gi
    k += size
df["GroupRand"] = df.Group.map(mapping)
print("control groups:", df.GroupRand.nunique(), "| pattern matched to GroupX")

control groups: 168 | pattern matched to GroupX


## 3. Repeated-seed cross-validation

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                             f1_score, brier_score_loss)
from xgboost import XGBClassifier

HP = dict(n_estimators=877, max_depth=14, learning_rate=0.023722174554030036,
          min_child_weight=2, colsample_bytree=0.5655509178160505,
          subsample=0.5482175662874387, gamma=0.0012682360748828933,
          reg_lambda=0.3024802937837069, reg_alpha=0.04238171400820899,
          random_state=42)
RUNTIME = dict(tree_method="hist", eval_metric="logloss",
               **({"device": "cuda"} if GPU else {"n_jobs": -1}))

PLAN = [("A  new condition, known protein", "RecordID",  [1, 2]),
        ("B  new protein",                  "SeqID",     [1, 2]),
        ("C  new family (their groups)",    "Group",     [1, 2, 3, 4, 5]),
        ("C* new family (hardened)",        "GroupX",    [1, 2, 3, 4, 5]),
        ("control  random merge",           "GroupRand", [1, 2, 3, 4, 5])]

raw = {}            # name -> list of [seed, roc, pr, acc, f1, brier]
t_all = time.time()
for name, key, seeds in PLAN:
    g, rows = df[key].to_numpy(), []
    for s in seeds:
        for i, j in StratifiedGroupKFold(5, shuffle=True, random_state=s).split(X, y, groups=g):
            m = XGBClassifier(**HP, **RUNTIME)
            m.fit(X[i], y[i], sample_weight=w[i])
            p = m.predict_proba(X[j])[:, 1]
            if len(np.unique(y[j])) < 2: continue
            rows.append([s,
                         roc_auc_score(y[j], p, sample_weight=w[j]),
                         average_precision_score(y[j], p, sample_weight=w[j]),
                         accuracy_score(y[j], p > .5, sample_weight=w[j]),
                         f1_score(y[j], p > .5, sample_weight=w[j]),
                         brier_score_loss(y[j], p, sample_weight=w[j])])
        json.dump({k2: v for k2, v in raw.items()} | {name: rows}, open("raw_folds.json", "w"))
    raw[name] = rows
    a = np.array(rows)
    print(f"{name:34s} {len(rows):3d} folds  ROC-AUC {a[:,1].mean():.3f} "
          f"(sd {a[:,1].std():.3f}, SE {a[:,1].std()/np.sqrt(len(a)):.3f})", flush=True)
print(f"\ntotal {(time.time()-t_all)/60:.0f} min")

A  new condition, known protein     10 folds  ROC-AUC 0.834 (sd 0.008, SE 0.003)
B  new protein                      10 folds  ROC-AUC 0.786 (sd 0.017, SE 0.005)
C  new family (their groups)        25 folds  ROC-AUC 0.683 (sd 0.045, SE 0.009)
C* new family (hardened)            25 folds  ROC-AUC 0.659 (sd 0.047, SE 0.009)
control  random merge               25 folds  ROC-AUC 0.701 (sd 0.037, SE 0.007)

total 45 min


## 4. Results table

In [ ]:
tab = {}
for name, rows in raw.items():
    a = np.array(rows); n = len(a)
    tab[name] = {"folds": n,
                 "ROC-AUC": f"{a[:,1].mean():.3f} ± {a[:,1].std()/np.sqrt(n):.3f}",
                 "PR-AUC":  f"{a[:,2].mean():.3f}",
                 "Accuracy":f"{a[:,3].mean():.3f}",
                 "F1":      f"{a[:,4].mean():.3f}",
                 "Brier":   f"{a[:,5].mean():.3f}"}
tab = pd.DataFrame(tab).T
print(tab.to_string())
print("\n(± is the standard error of the mean, not the fold spread)")
tab.to_csv("repeated_benchmark_results.csv")

                                folds        ROC-AUC PR-AUC Accuracy     F1  Brier
A  new condition, known protein    10  0.834 ± 0.003  0.848    0.759  0.785  0.172
B  new protein                     10  0.786 ± 0.005  0.804    0.719  0.736  0.204
C  new family (their groups)       25  0.683 ± 0.009  0.714    0.625  0.649  0.248
C* new family (hardened)           25  0.659 ± 0.009  0.685    0.606  0.608  0.258
control  random merge              25  0.701 ± 0.007  0.725    0.649  0.668  0.236

(± is the standard error of the mean, not the fold spread)


## 5. Is the C-C* gap real?

Two comparisons decide it:

- **C\* vs control** — both have 168 groups, so coarsening is held constant and
  only homology differs. This is the comparison that matters.
- **C vs control** — if these agree, coarsening costs nothing on its own.

In [ ]:
from scipy import stats
def arr(n): return np.array(raw[n])[:, 1]
pairs = [("C  new family (their groups)", "C* new family (hardened)"),
         ("control  random merge",        "C* new family (hardened)"),
         ("C  new family (their groups)", "control  random merge")]
for a_name, b_name in pairs:
    a, b = arr(a_name), arr(b_name)
    d = a.mean() - b.mean()
    se = np.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))
    t, p = stats.ttest_ind(a, b, equal_var=False)
    print(f"{a_name[:28]:30s} vs {b_name[:26]:28s}  diff {d:+.3f}  SE {se:.3f}  t {t:+.2f}  p {p:.3f}")

print("\nreading it:")
print("  control vs C* significant  -> the hardened grouping removes real leakage")
print("  C vs control not significant -> coarsening alone costs nothing")

C  new family (their groups)   vs C* new family (hardened)      diff +0.024  SE 0.013  t +1.78  p 0.081
control  random merge          vs C* new family (hardened)      diff +0.042  SE 0.012  t +3.45  p 0.001
C  new family (their groups)   vs control  random merge         diff -0.018  SE 0.012  t -1.54  p 0.131

reading it:
  control vs C* significant  -> the hardened grouping removes real leakage
  C vs control not significant -> coarsening alone costs nothing


## 6. Per-seed stability check

In [ ]:
for name in ["C  new family (their groups)", "C* new family (hardened)", "control  random merge"]:
    a = np.array(raw[name])
    per = [a[a[:,0]==s][:,1].mean() for s in np.unique(a[:,0])]
    print(f"{name:34s} per-seed means: {np.round(per,3).tolist()}")
print("\nIf per-seed means swing widely, the partition - not the model - dominates,")
print("and even 25 folds may understate the uncertainty. Report the spread either way.")

C  new family (their groups)       per-seed means: [0.682, 0.694, 0.695, 0.664, 0.678]
C* new family (hardened)           per-seed means: [0.648, 0.653, 0.665, 0.685, 0.643]
control  random merge              per-seed means: [0.714, 0.702, 0.696, 0.691, 0.703]

If per-seed means swing widely, the partition - not the model - dominates,
and even 25 folds may understate the uncertainty. Report the spread either way.


In [ ]:
from google.colab import files
files.download("repeated_benchmark_results.csv")
files.download("raw_folds.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>